# Lab 2 — Probe a real V-JEPA 2 checkpoint

In this lab we will **not train** V-JEPA 2. We will use a released checkpoint to connect the architecture from the slides to real tensors and predictions.

### Three questions with answers to keep in mind

1. **What can this classifier actually predict?**  
   The checkpoint uses a **Something-Something-v2 (SSv2)** classification head with 174 fine-grained action labels.

2. **Which output is the reusable representation, and which output belongs to the downstream task?**  
   We will compare the encoder's hidden states with the final SSv2 logits.

3. **Does changing temporal order change the representation?**  
   We will compare the same clip played forward and backward.

> **Important:** the sample clip is a *bowling* clip from Kinetics, but **"bowling" is not an SSv2 class**.  
> The classifier is therefore forced to choose among the SSv2 hand/object interaction labels it knows. This is useful: it lets us separate **representation quality** from the **label space of a particular downstream head**.



In [ ]:

%pip install -q "transformers>=4.53,<5" "huggingface_hub>=0.30" "av>=12"


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import av
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForVideoClassification, AutoVideoProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "facebook/vjepa2-vitl-fpc16-256-ssv2"
# This is the same Kinetics-mini bowling clip used in the official HF example
VIDEO_URL = (
    "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/"
    "val/bowling/-WH-lxmGJVY_000005_000015.mp4"
)
print("Device:", DEVICE)


## 1. Load V-JEPA 2 + the SSv2 classification head

This checkpoint contains two conceptually different pieces:

$
\text{video}
\;\xrightarrow{\text{V-JEPA 2 encoder}}\;
\text{latent tokens}
\;\xrightarrow{\text{attentive classifier}}\;
174\ \text{SSv2 logits}.
$

The **hidden states** come from the video encoder and are the representations we would reuse for probing or transfer.

The **logits** are task-specific outputs from the SSv2 classification head.


In [ ]:
# download the checkpoint
dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
processor = AutoVideoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForVideoClassification.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
).to(DEVICE).eval()

parameters = sum(p.numel() for p in model.parameters())
print(f"Parameters: {parameters / 1e6:.1f}M")
print("Expected frames per clip:", model.config.frames_per_clip)
print("Labels:", model.config.num_labels)


## 2. Before inference: what labels are even possible?

Something-Something-v2 is **not** a generic list of activities such as *bowling*, *running*, or *playing football*.

Its 174 classes emphasize **fine-grained interactions and temporal relations**. Representative types include:

- **direction:** moving something up/down, toward/away from the camera;
- **placement:** putting something into/onto/under something;
- **directional manipulation:** pushing or pulling left-to-right vs. right-to-left;
- **physical outcome:** an object falls, slides, rolls, spills, or stays in place;
- **intent/failure:** pretending to perform an action or failing to complete it.

These labels are especially interesting for a video model because many pairs cannot be distinguished reliably from a single static frame.


In [ ]:
print(model.config.id2label)

## 3. Decode and sample the video

We uniformly sample the number of frames expected by the checkpoint.

Before inference, visually inspect the clip: **what motion or object interaction could an SSv2 classifier latch onto, even though "bowling" itself is unavailable?**


In [ ]:
video_path = Path("/tmp/jepa_tutorial_sample.mp4")

if not video_path.exists():
    urlretrieve(VIDEO_URL, video_path)

with av.open(str(video_path)) as container:
    decoded = [
        frame.to_ndarray(format="rgb24")
        for frame in container.decode(video=0)
    ]

decoded = torch.from_numpy(np.stack(decoded)).permute(0, 3, 1, 2)  # T×C×H×W

indices = torch.linspace(
    0,
    len(decoded) - 1,
    steps=model.config.frames_per_clip,
).round().long()

video = decoded[indices]

print("Decoded frames:", len(decoded))
print("Selected frames:", len(video))
print("Video tensor [T,C,H,W]:", tuple(video.shape), video.dtype)

# Show 8 frames in temporal order.
shown = np.linspace(0, len(video) - 1, 8).round().astype(int)

fig, axes = plt.subplots(1, len(shown), figsize=(14, 2.4))

for ax, i in zip(axes, shown):
    frame = video[i].permute(1, 2, 0).cpu().numpy()
    ax.imshow(frame)
    ax.set_title(f"t={i}")
    ax.axis("off")

plt.suptitle("Uniformly sampled frames — forward temporal order")
plt.tight_layout()
plt.show()

## 4. Run inference and inspect the representation

We request `output_hidden_states=True` so that one forward pass gives us both:

- `outputs.logits`: the **174-dimensional SSv2 classifier output**;
- `outputs.hidden_states`: the **encoder representations across depth**.


In [ ]:
inputs = processor(video, return_tensors="pt")
inputs = {
    name: value.to(DEVICE)
    for name, value in inputs.items()
}

with torch.inference_mode():
    outputs = model(
        **inputs,
        output_hidden_states=True,
    )

print("Processed model inputs:")
print({name: tuple(value.shape) for name, value in inputs.items()})

print("\nTask-specific output:")
print("  logits [batch, classes]:", tuple(outputs.logits.shape))

print("\nEncoder representation:")
print("  number of hidden-state tensors:", len(outputs.hidden_states))
print(
    "  last hidden state [batch, tokens, dim]:",
    tuple(outputs.hidden_states[-1].shape),
)

### What should we conclude from these tensors?

**1. The logits are not "the V-JEPA representation."**  
They are the output of a classifier trained specifically for 174 SSv2 classes.

**2. The hidden states are the reusable part.**  
They contain token-level video features produced by the encoder and can be used for linear probes, transfer learning, dense prediction, similarity analysis, or another downstream head.

**3. The representation contains much more information than the 174 logits.**  
A downstream classifier compresses a large set of latent tokens into one task-specific decision.


## 5. What does the SSv2 head predict for this out-of-label-space clip?

We now inspect the top-5 classes.

Remember:

> **"Bowling" is impossible as an output.**

Therefore, the result is best interpreted as the nearest available SSv2 action hypotheses — **not** as an open-vocabulary description of the clip.


In [ ]:
def topk_predictions(logits, k=5):
    probs = logits.float().softmax(dim=-1)[0]
    values, indices = probs.topk(k)

    return [
        (
            model.config.id2label[index.item()],
            value.item(),
        )
        for value, index in zip(values, indices)
    ]

forward_top5 = topk_predictions(outputs.logits, k=5)

print("Forward top-5 SSv2 predictions:\n")

for rank, (label, probability) in enumerate(forward_top5, start=1):
    print(f"{rank}. {probability:6.2%}  {label}")

labels = [label for label, _ in forward_top5]
values = [prob for _, prob in forward_top5]

plt.figure(figsize=(9, 3.6))
plt.barh(labels[::-1], values[::-1])
plt.xlabel("softmax probability")
plt.title("V-JEPA 2 + SSv2 attentive classifier")
plt.tight_layout()
plt.show()

### Discussion of the classification result

There are two different questions we could ask:

**Bad question for this checkpoint:**  
> "Did V-JEPA 2 correctly recognize bowling?"

The classifier literally has no *bowling* output class, so this experiment cannot answer that question.

**Better question:**  
> "Given an SSv2 action vocabulary, which temporal interaction patterns does the downstream head associate with this video?"

## 6. Temporal-order probe: forward vs. reversed video

Now we keep **exactly the same frames** but reverse their order.

We will compare:

1. the top-5 SSv2 predictions;
2. the cosine similarity between mean-pooled hidden representations;

> This is a **single-clip diagnostic**, not a benchmark and not proof of physical understanding.


In [ ]:
reversed_video = torch.flip(video, dims=[0])

reversed_inputs = processor(
    reversed_video,
    return_tensors="pt",
)
reversed_inputs = {
    name: value.to(DEVICE)
    for name, value in reversed_inputs.items()
}

with torch.inference_mode():
    reversed_outputs = model(
        **reversed_inputs,
        output_hidden_states=True,
    )

reverse_top5 = topk_predictions(reversed_outputs.logits, k=5)

print("FORWARD")
for rank, (label, probability) in enumerate(forward_top5, start=1):
    print(f"{rank}. {probability:6.2%}  {label}")

print("\nREVERSED")
for rank, (label, probability) in enumerate(reverse_top5, start=1):
    print(f"{rank}. {probability:6.2%}  {label}")

forward_embedding = outputs.hidden_states[-1].float().mean(dim=1)
reverse_embedding = reversed_outputs.hidden_states[-1].float().mean(dim=1)

final_cosine = F.cosine_similarity(
    forward_embedding,
    reverse_embedding,
).item()

forward_top1 = forward_top5[0][0]
reverse_top1 = reverse_top5[0][0]

forward_set = {label for label, _ in forward_top5}
reverse_set = {label for label, _ in reverse_top5}
top5_overlap = len(forward_set & reverse_set)

print("\n--- Summary ---")
print(f"Final-layer mean-pooled cosine: {final_cosine:.4f}")
print("Top-1 changed:", forward_top1 != reverse_top1)
print(f"Top-5 label overlap: {top5_overlap}/5")

Temporal reversal perturbs the representation enough to change the downstream score distribution, but not enough to radically change the global semantic representation of this clip. We have a small sample in this case, but it already shows that the representations are not 100% the same.